In [ ]:
#@title Run this cell first -- homework checking setup { display-mode: "form" }
#@markdown **Run this cell before anything else**, every time you open the notebook.
#@markdown <br><br>It sets up `grader.check(...)` so you can check your answers as you go.
#@markdown Nothing is installed and nothing is downloaded -- this notebook is
#@markdown self-contained, so it works even if Colab or GitHub is having a bad day.

import base64
import datetime
import doctest
import json
import sys

RECEIPT_PREFIX = "CS1910-RECEIPT:"

_HTML = """<div style="font-family:system-ui,sans-serif;font-size:14px;margin:4px 0;
 padding:10px 12px;border-left:5px solid {bar};background:{bg};border-radius:4px">
 <b>{name}</b> &mdash; {earned}/{possible} {verdict}{detail}</div>"""


class Result:
    """One question's outcome. Displays as HTML in Colab, as text everywhere else.

    The text form is what lands in the notebook's saved output and what the grader
    parses, so keep the first line machine-readable.
    """

    def __init__(self, name, earned, possible, messages):
        self.name, self.earned, self.possible = name, earned, possible
        self.messages = messages

    @property
    def passed(self):
        return self.earned == self.possible

    def __repr__(self):
        head = "{}: {}/{} {}".format(self.name, self.earned, self.possible,
                                     "PASSED" if self.passed else "FAILED")
        return "\n".join([head] + self.messages)

    def _repr_html_(self):
        bar, bg = ("#2e7d32", "#edf7ed") if self.passed else ("#c62828", "#fdecea")
        verdict = "passed" if self.passed else "not passing yet"
        detail = ""
        if self.messages:
            body = "\n".join(self.messages).replace("&", "&amp;").replace("<", "&lt;")
            detail = ('<pre style="margin:8px 0 0;white-space:pre-wrap;font-size:12px;'
                      'color:#444">{}</pre>'.format(body))
        return _HTML.format(bar=bar, bg=bg, name=self.name, earned=self.earned,
                            possible=self.possible, verdict=verdict, detail=detail)


def run_case(code, globs):
    """Run one doctest string. Returns (passed, report_text).

    `get_doctest` copies `globs`, so a case that rebinds a name cannot leak that
    change back into the student's namespace or into the next case.
    """
    parser = doctest.DocTestParser()
    test = parser.get_doctest(code.rstrip() + "\n", dict(globs), "case", None, 0)
    runner = doctest.DocTestRunner(verbose=False,
                                   optionflags=doctest.NORMALIZE_WHITESPACE)
    out = []
    try:
        runner.run(test, out=out.append, clear_globs=False)
    except Exception as exc:                      # a case that raises is a failure
        return False, "{}: {}".format(type(exc).__name__, exc)
    return runner.failures == 0, "".join(out)


def score(name, spec, globs):
    """Score one question against `globs`. Returns a Result."""
    cases = spec["suites"][0]["cases"] if "suites" in spec else spec["cases"]
    per_case = bool(cases) and all("points" in c for c in cases)

    earned = possible = 0
    failures, notes = [], []
    for case in cases:
        ok, report = run_case(case["code"], globs)
        if per_case:
            possible += case["points"]
            earned += case["points"] if ok else 0
        if ok:
            if case.get("success_message"):
                notes.append(case["success_message"])
        else:
            failures.append(case.get("failure_message") or _trim(report))

    if not per_case:
        possible = spec.get("points", len(cases))
        earned = 0 if failures else possible

    return Result(name, earned, possible, failures or notes)


def _trim(report, limit=600):
    """Doctest reports are verbose; keep the part a first-year student can act on."""
    text = report.strip()
    return text if len(text) <= limit else text[:limit] + "\n... (truncated)"


def caller_namespace():
    """Where the student's variables live.

    In Colab and Jupyter that is the interactive namespace, which is right even when
    `check()` is called from inside a function. Outside IPython -- the headless grader,
    or a plain `python` run -- fall back to the calling frame's globals, which is the
    namespace `exec` was handed.
    """
    try:
        shell = get_ipython()           # noqa: F821  (injected by IPython)
        if shell is not None:
            return shell.user_ns
    except NameError:
        pass
    return sys._getframe(2).f_globals


class Grader:
    """Holds the tests for one homework and remembers what has been checked.

    Named `grader` in the student notebook so existing `grader.check("q1")` cells
    keep working untouched.
    """

    def __init__(self, hw, tests):
        self.hw, self.tests, self.log = hw, tests, {}

    def check(self, name, globs=None):
        if name not in self.tests:
            raise KeyError("no test named {!r} in {} (have: {})".format(
                name, self.hw, ", ".join(sorted(self.tests))))
        result = score(name, self.tests[name], globs or caller_namespace())
        self.log[name] = (result.earned, result.possible)
        return result

    def check_all(self, globs):
        """Score every question against one namespace. This is what grading uses."""
        results = [score(n, self.tests[n], globs) for n in sorted(self.tests)]
        self.log = {r.name: (r.earned, r.possible) for r in results}
        return results

    def summary(self):
        return {"hw": self.hw,
                "results": {k: list(v) for k, v in sorted(self.log.items())},
                "total": sum(e for e, _ in self.log.values()),
                "possible": sum(p for _, p in self.log.values())}

    def receipt(self, student_id="", extra=None):
        """Print the one-line machine-readable receipt and return it.

        The grader re-runs the tests itself, so this line is a convenience for the
        student and a fast triage signal -- never the authoritative score.
        """
        body = self.summary()
        body["id"] = str(student_id)
        body["t"] = datetime.datetime.now().isoformat(timespec="seconds")
        if extra:
            body.update(extra)
        blob = base64.b64encode(
            json.dumps(body, sort_keys=True, separators=(",", ":")).encode()
        ).decode()
        line = RECEIPT_PREFIX + blob
        print(line)
        return line


def parse_receipt(text):
    """Pull the last receipt out of arbitrary text. Returns a dict, or None."""
    found = None
    for chunk in text.split(RECEIPT_PREFIX)[1:]:
        token = chunk.split()[0] if chunk.split() else ""
        try:
            found = json.loads(base64.b64decode(token))
        except Exception:
            continue
    return found


def load_ok_tests(directory):
    """Read a directory of `q*.py` test files (OK_FORMAT doctests) into {name: spec}."""
    import pathlib
    tests = {}
    for path in sorted(pathlib.Path(directory).glob("*.py")):
        namespace = {}
        exec(compile(path.read_text(encoding="utf-8"), str(path), "exec"), namespace)
        spec = namespace["test"]
        tests[spec["name"]] = spec
    return tests

_TESTS = {'q1': {'name': 'q1',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> def t1():\n'
                                       "...     n = write_log('_hw14_a.txt', [('tiktok', 95), "
                                       "('yt', 40)])\n"
                                       "...     with open('_hw14_a.txt') as f:\n"
                                       '...         text = f.read()\n'
                                       '...     return n == 2 and text == '
                                       "'tiktok,95\\nyt,40\\n'\n"
                                       '>>> t1()\n'
                                       'True',
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> def t1b():\n'
                                       "...     write_log('_hw14_a.txt', [('a', 1), ('b', 2), "
                                       "('c', 3)])\n"
                                       "...     write_log('_hw14_a.txt', [('z', 9)])\n"
                                       "...     with open('_hw14_a.txt') as f:\n"
                                       "...         return f.read() == 'z,9\\n'\n"
                                       '>>> t1b()\n'
                                       'True',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q2': {'name': 'q2',
        'points': 2,
        'suites': [{'cases': [{'code': '>>> def t2():\n'
                                       "...     with open('_hw14_b.txt', 'w') as f:\n"
                                       "...         f.write('tiktok,95\\nyt,40\\n')\n"
                                       "...     return read_log('_hw14_b.txt')\n"
                                       '>>> t2()\n'
                                       "[('tiktok', 95), ('yt', 40)]",
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> def t2b():\n'
                                       "...     with open('_hw14_c.txt', 'w') as f:\n"
                                       "...         f.write('ig,12\\n\\n')\n"
                                       "...     return read_log('_hw14_c.txt')\n"
                                       '>>> t2b()\n'
                                       "[('ig', 12)]",
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> def t2c():\n'
                                       "...     with open('_hw14_d.txt', 'w') as f:\n"
                                       "...         f.write('  tiktok,95  \\nyt,40\\n')\n"
                                       "...     return read_log('_hw14_d.txt')\n"
                                       '>>> t2c()\n'
                                       "[('tiktok', 95), ('yt', 40)]",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q3': {'name': 'q3',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> total_minutes([('tiktok', 95), ('yt', 40)])\n135",
                               'hidden': False,
                               'locked': False},
                              {'code': '>>> total_minutes([])\n0',
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q4': {'name': 'q4',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> worst_app([('a', 10), ('b', 50), ('c', 5)])\n'b'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> worst_app([('solo', 7)])\n'solo'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> worst_app([])\n''",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q5': {'name': 'q5',
        'points': 2,
        'suites': [{'cases': [{'code': ">>> summary_line([('tiktok', 95), ('yt', 40)])\n"
                                       "'tiktok=95,yt=40'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> summary_line([('a', 1)])\n'a=1'",
                               'hidden': False,
                               'locked': False},
                              {'code': ">>> summary_line([])\n''",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q6': {'name': 'q6',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q6_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]},
 'q7': {'name': 'q7',
        'points': 1,
        'suites': [{'cases': [{'code': ">>> q7_option.strip().upper() == 'B'\nTrue",
                               'hidden': False,
                               'locked': False}],
                    'scored': True,
                    'setup': '',
                    'teardown': '',
                    'type': 'doctest'}]}}

grader = Grader('14-File-IO', _TESTS)
check = grader.check

print("Ready. 14-File-IO loaded: 7 question(s), 12 point(s).")
print("Check a question any time with, for example:  grader.check(\"q1\")")

## Before you begin

### Submission
Before you start working on this Homework:

1. Press **File->Save a copy in Drive** from the menu. This will save your own copy of this notebook

2. Follow the steps below and ensure you run the cells in order from the top. (You'll have to ensure you run any cells above this one too)

3. You should get into the habit of saving your file occassionally as you work

4. Certain cells will check your work against the expected answer, when you run these cells you'll know if your work is correct.

5. After all the changes are done and progress is saved **File->Download->Download.ipynb** and then upload the .ipynb file to moodle

# Homework: File I/O — Screen Time Export

Your phone knows exactly how long you spent on each app last week, and it will export that
for you as a text file. It looks like this:

```
tiktok,95
instagram,48
youtube,40
messages,22
```

One app per line, a comma, then the minutes. No headings, no formatting — just text.

That format is not a coincidence. It is boring on purpose, because **every** program can
read it. This homework is about the two things your program has to do with a file: get the
data out of it, and put results back.

One thing to know before you start. Everything your program has in variables lives in
**RAM**, which is wiped the moment the program stops. That is why closing a notebook loses
your variables. A file is **storage** — it survives. Reading and writing files is how a
program remembers anything at all.

> **A Colab note:** the file browser in the left sidebar is a temporary disk that Colab
> throws away when your session ends. Files you write there are real while you are working
> and gone tomorrow. That is fine for this homework — everything you need, you create.

## Question 1 — Write the log out

Write `write_log(path, rows)`. `rows` is a list of `(app, minutes)` tuples. It writes one
line per row in the form `app,minutes`, **each line ending with a newline**, and returns
how many rows it wrote.

```python
write_log('screentime.txt', [('tiktok', 95), ('yt', 40)])   # returns 2
```

leaves the file holding exactly:

```
tiktok,95
yt,40
```

Two details carry the marks:

- **`f.write()` adds nothing of its own.** Unlike `print()`, it writes exactly the string
  you hand it — so the `\n` at the end of each line has to be in that string, including on
  the last line.
- **Opening in `'w'` mode empties the file first.** Calling `write_log` twice leaves only
  the second set of rows, and the second test checks exactly that.

Use the `with` form, which closes the file for you even if something goes wrong in the
middle:

```python
with open(path, 'w') as f:
    f.write("something\n")
```

`minutes` is an `int`, and you cannot write a number to a text file — everything in a text
file is text. `f"{app},{minutes}\n"` converts it for you.

In [ ]:
def write_log(path, rows):
    ...

# Uncomment the last two lines once write_log works -- until it does, there is no
# screentime.txt to open, and every question below reads the file it creates.
write_log('screentime.txt', [('tiktok', 95), ('instagram', 48), ('youtube', 40)])
# with open('screentime.txt') as f:
#     print(f.read())

In [ ]:
grader.check("q1")

## Question 2 — Read the log back in

Write `read_log(path)`, which reads a file in that format and returns a list of
`(app, minutes)` tuples with **minutes as an `int`**.

```
a file holding  'tiktok,95\nyt,40\n'   ->  [('tiktok', 95), ('yt', 40)]
a file holding  'ig,12\n\n'            ->  [('ig', 12)]      the blank line is skipped
a file holding  '  tiktok,95  \nyt,40\n'  ->  [('tiktok', 95), ('yt', 40)]
```

Looping over an open file gives you one line at a time:

```python
with open(path) as f:
    for line in f:
        ...
```

Three things happen to each line, in order:

1. **`.strip()` the whole line, before you split it.** Every line still has its `\n` on
   the end — that is how the file records where lines break — and this export sometimes
   has spaces around a line too. Strip first and the rest is clean.

   This is the step people skip, because it usually seems to work anyway: `int('95\n')`
   happens to succeed, since `int()` tolerates surrounding whitespace. Then one day the
   untidy field is a name rather than a number, `'youtube\n' == 'youtube'` is `False`,
   and the afternoon is gone. Strip the line.

   `.strip()` also turns a blank line into the empty string, which is how you spot one
   and skip it.
2. **`.split(',')` it.** That gives you a list of the pieces: `'tiktok,95'` becomes
   `['tiktok', '95']`.
3. **`int()` the minutes.** Everything read from a text file is a `str`, including `'95'`.
   Nothing is a number until you make it one.

In [ ]:
def read_log(path):
    ...

# try it out -- reads the file Question 1 wrote
print(read_log('screentime.txt'))

In [ ]:
grader.check("q2")

## Question 3 — Total the minutes

Write `total_minutes(rows)`, which returns the total number of minutes across every row.

```
total_minutes([('tiktok', 95), ('yt', 40)])  ->  135
total_minutes([])                            ->  0
```

The minutes are already `int`s if Question 2 did its job, so this is addition — but only
because you converted them back there. Try it for a moment on rows where the minutes are
still strings and watch `'95' + '40'` give you `'9540'`. That is the entire reason step 3
of Question 2 exists.

In [ ]:
def total_minutes(rows):
    ...

# try it out
print(total_minutes(read_log('screentime.txt')))

In [ ]:
grader.check("q3")

## Question 4 — The worst offender

Write `worst_app(rows)`, which returns the **name** of the app with the most minutes. If
there are no rows at all, return the empty string `''`.

```
worst_app([('a', 10), ('b', 50), ('c', 5)])  ->  'b'
worst_app([('solo', 7)])                     ->  'solo'
worst_app([])                                ->  ''
```

This is the "largest so far" pattern, and it comes up constantly: keep track of the best
you have seen, compare each new one against it, and replace it when you find better. You
need to remember two things as you go — the best minutes *and* which app had them.

In [ ]:
def worst_app(rows):
    ...

# try it out
print(worst_app(read_log('screentime.txt')))

In [ ]:
grader.check("q4")

## Question 5 — One line, joined

Write `summary_line(rows)`, which squashes the whole week into a single string:

```
summary_line([('tiktok', 95), ('yt', 40)])  ->  'tiktok=95,yt=40'
summary_line([('a', 1)])                    ->  'a=1'
summary_line([])                            ->  ''
```

Note where the commas are: **between** the pieces, and not on the end. That is exactly what
`.join()` is for, and it is why building a string by repeatedly adding `piece + ","` always
leaves you trimming a trailing comma off afterwards.

`.join()` is called on the separator, which reads backwards the first few times you see it:

```python
",".join(['a', 'b', 'c'])   # 'a,b,c'
"-".join(['2026'])          # '2026'   -- one piece, no separator needed
",".join([])                # ''
```

It joins **strings only** — hand it a number and you get a `TypeError`. So you need a list
of ready-made `'app=minutes'` strings first, which a comprehension builds in one line:

```python
[f"{app}={minutes}" for app, minutes in rows]
```

Then join that. The empty case falls out on its own — no special handling needed.

In [ ]:
def summary_line(rows):
    ...

# try it out
print(summary_line(read_log('screentime.txt')))

In [ ]:
grader.check("q5")

## Question 6 — `'w'` or `'a'`?

You want to add this week's row to a file that already holds the previous eleven weeks,
without destroying them.

Which mode do you open the file in?

**A** — `'w'`

**B** — `'a'`

**C** — `'r'`

Set `q6_option` to `'A'`, `'B'` or `'C'`.

Worth being certain about, because getting it wrong is not an error message — it is a file
that silently used to have your data in it. `'w'` truncates the file to nothing the moment
you open it, before you have written a single byte.

In [ ]:
q6_option = ...

In [ ]:
grader.check("q6")

## Question 7 — Where does the file actually live?

Your code says:

```python
open('data/screentime.txt')
```

Is that an absolute path or a relative one?

**A** — absolute: it names one exact location on the machine

**B** — relative: it is resolved from wherever the program is currently running

**C** — neither; that is not a valid path

Set `q7_option` to `'A'`, `'B'` or `'C'`.

An **absolute** path starts from the root of the file system — `/home/chris/data/x.txt` on
macOS or Linux, `C:\Users\chris\data\x.txt` on Windows. A **relative** path starts from
the current working directory, and `.` means "here" while `..` means "the folder above".

This is the difference between code that runs on your laptop and code that runs on your
marker's, and it is why almost all of the "but it worked for me" bugs in this course are
path bugs.

In [ ]:
q7_option = ...

In [ ]:
grader.check("q7")

## Beyond the homework

Not graded:

- `f.readlines()` reads every line into a list in one go, instead of looping over the file.
  Rewrite `read_log` with it. When would reading the whole file at once be a bad idea?
- Write `append_week(path, rows)` using `'a'` mode, then a `weekly_report(path)` that reads
  the accumulated file and prints a total per app. You will want a dictionary.
- Windows separates folders with `\` and everything else with `/`. Look up
  `os.path.join()` and rewrite `'data/screentime.txt'` so it is right on both.
- Save a log to your Google Drive so it is still there next week. `from google.colab import
  drive` is the start of it.

# Submission

Save a copy in Drive to ensure you have your own copy of this notebook and to ensure that you've saved your latest work.

- Ensure you run all the cells in order from the top to the bottom
- Run the final cell to see your total and produce your receipt
- **File->Download->Download .ipynb** and upload the `.ipynb` file to Moodle

Your mark comes from your instructor re-running this notebook, not from the receipt
line -- so make sure the notebook itself is saved with your work in it.

In [ ]:
#@title Finished? Run this to see your total { display-mode: "form" }
#@markdown Re-checks every question and prints your score, plus a receipt line.
#@markdown <br><br>Your grade comes from your instructor re-running this notebook,
#@markdown so the receipt is for your records -- make sure the notebook itself is saved.

student_id = ""  #@param {type:"string"}

for _r in grader.check_all(globals()):
    print(_r)
_s = grader.summary()
print("\nTOTAL: {}/{}".format(_s["total"], _s["possible"]))
grader.receipt(student_id)
